# 02 — Patient-level training, validation and held-out testing

This notebook runs the consolidated MSD experiment. It trains the pixel-wise Random Forest at 16×16, 32×32 and native 240×240 and compares it with Otsu, K-means and watershed baselines. Metrics are computed at patient-held-out slice level. The split is patient-level to avoid slice leakage.

**Full study setting:** `MAX_PATIENTS = 0` uses all available labelled cases. This may take substantial CPU time. Set a smaller cap only for a pilot; clearly label any capped subset in research reporting.

In [ ]:
from pathlib import Path
import sys, json, time
import pandas as pd
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
DATA_ROOT = ROOT / "data/raw/Task01_BrainTumour"
MAX_PATIENTS = 0       # 0 = all labelled cases; use 120 for a pilot run
SLICES_PER_PATIENT = 8
SAMPLES_PER_CLASS = 250
TREES = 120
SEED = 42
RESOLUTIONS = (16, 32, 240)

In [ ]:
from qmi.experiments.runner import run_msd_experiment
start = time.perf_counter()
results = run_msd_experiment(DATA_ROOT, max_patients=MAX_PATIENTS,
    max_slices_per_patient=SLICES_PER_PATIENT, samples_per_class=SAMPLES_PER_CLASS,
    trees=TREES, seed=SEED, resolutions=RESOLUTIONS, output_root=ROOT)
print(f"Total wall time: {time.perf_counter()-start:.1f}s")
print(json.dumps({"dataset": results["dataset"], "patients": results["selected_patients"],
    "resolutions": list(results["resolutions"])}, indent=2))

In [ ]:
REPORTS = ROOT / "outputs/reports"
for resolution in RESOLUTIONS:
    print(f"\nHeld-out test summary — {resolution}×{resolution}")
    display(pd.read_csv(REPORTS / f"msd_test_summary_{resolution}.csv"))
    print("Patient-level split file:", REPORTS / "msd_patient_split.json")

## Saved deliverables

- `artifacts/msd_rf_16.joblib`, `artifacts/msd_rf_32.joblib`, `artifacts/msd_rf_240.joblib` — trained models
- `artifacts/msd_rf_16.json`, `artifacts/msd_rf_32.json` — provenance and configuration
- `outputs/reports/msd_patient_split.json` — exact patient IDs in each partition
- `outputs/reports/msd_validation_*.csv` — validation metrics
- `outputs/reports/msd_test_*.csv` — held-out test metrics
- `outputs/reports/msd_experiment_summary.json` — combined machine-readable summary
- `db/project_quantum.sqlite3` — experiment run history

Do not use the validation set to report final performance. The test set is held out from model/baseline fitting.